## Classifier improvements

swap AdaBoost for histogram-based gradient boosting, validate with 10-fold cross-validation, then run the Punzi scan on the new classifier and write its efficiencies back into `bdt_results.json`. `mass_fit.ipynb` then computes the discovery time at this working point.

same 7 raw inputs as `bdt.ipynb` (MASS excluded), no feature engineering.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import pickle
import os
from sklearn.model_selection import KFold
from sklearn.ensemble import AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

os.makedirs("../plots", exist_ok=True)

In [ ]:
features  = ["PT1", "PT2", "P1", "P2", "TotalPT", "VertexChisq", "Isolation", "MASS"]
features7 = [f for f in features if f != "MASS"]

signal = pd.read_csv("../data/signal_Bs2MuMu.txt", sep=r"\s+", header=None, names=features)
background = pd.read_csv("../data/background_combinatorial.txt", sep=r"\s+", header=None, names=features)
signal = signal.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
background = background.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

X = pd.concat([signal[features7], background[features7]], axis=0).reset_index(drop=True)
y = np.concatenate([np.ones(len(signal)), np.zeros(len(background))])

In [ ]:
def make_adaboost():
    base = DecisionTreeClassifier(max_depth=2, min_samples_leaf=50, random_state=42)
    return AdaBoostClassifier(estimator=base, n_estimators=100, learning_rate=0.5, random_state=42)

def make_gradboost():
    return HistGradientBoostingClassifier(max_depth=4, max_iter=200, learning_rate=0.1,
                                          min_samples_leaf=50, random_state=42)

cv = KFold(n_splits=10, shuffle=True, random_state=42)
cv_results = {}
for name, factory in [("AdaBoost", make_adaboost), ("GradBoost", make_gradboost)]:
    accs = []
    for train_idx, test_idx in cv.split(X):
        clf = factory()
        clf.fit(X.iloc[train_idx], y[train_idx])
        accs.append(accuracy_score(y[test_idx], clf.predict(X.iloc[test_idx])))
    cv_results[name] = {"folds": [float(a) for a in accs],
                        "mean": float(np.mean(accs)),
                        "std":  float(np.std(accs))}
    print(f"{name:10s}: {np.mean(accs):.4f} +/- {np.std(accs):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
names = list(cv_results.keys())
means = [cv_results[n]["mean"] for n in names]
stds  = [cv_results[n]["std"]  for n in names]
x = np.arange(len(names))
ax.bar(x, means, yerr=stds, capsize=6, color=["steelblue", "darkorange"])
for xi, m, s in zip(x, means, stds):
    ax.text(xi, m + s + 0.003, f"{m:.4f}\n+/- {s:.4f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(min(means) - 0.03, max(means) + 0.02)
ax.set_ylabel("10-fold CV accuracy")
plt.tight_layout()
plt.savefig("../plots/cv_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### Punzi scan on the improved model

same FOM scan as in `punzi_fom.ipynb`, but on GradBoost trained on the full sample. write the resulting efficiencies and threshold into `bdt_results.json` so `mass_fit.ipynb` uses this working point when computing the discovery time.

In [ ]:
gb = make_gradboost()
gb.fit(X, y)
scores_sig = gb.predict_proba(signal[features7])[:, 1]
scores_bkg = gb.predict_proba(background[features7])[:, 1]

N_BKG_YEAR = 2000
n_sigma = 5.0
bkg_eff_floor = 1.0 / len(background)

thresholds = np.linspace(0.01, 0.99, 300)
fom_vals, sig_eff_scan, bkg_eff_scan = [], [], []
for t in thresholds:
    eps_s = (scores_sig >= t).mean()
    eps_b = max((scores_bkg >= t).mean(), bkg_eff_floor)
    B = N_BKG_YEAR * eps_b
    fom_vals.append(eps_s / (n_sigma/2 + np.sqrt(B)))
    sig_eff_scan.append(eps_s); bkg_eff_scan.append(eps_b)
fom_vals = np.array(fom_vals); sig_eff_scan = np.array(sig_eff_scan); bkg_eff_scan = np.array(bkg_eff_scan)

i_fom = int(np.argmax(fom_vals))
thr_punzi_imp = float(thresholds[i_fom])
sig_eff_imp_punzi = float(sig_eff_scan[i_fom])
bkg_eff_imp_punzi = float(bkg_eff_scan[i_fom])
print(f"improved Punzi: t={thr_punzi_imp:.3f}  sig_eff={sig_eff_imp_punzi:.4f}  bkg_eff={bkg_eff_imp_punzi:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, fom_vals, color="steelblue")
ax.axvline(thr_punzi_imp, color="black", linestyle="--",
           label=f"improved Punzi t = {thr_punzi_imp:.3f}")
ax.set_xlabel("GradBoost threshold"); ax.set_ylabel("Punzi FOM"); ax.legend()
plt.tight_layout()
plt.savefig("../plots/punzi_improved.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
with open("../improved_model.pkl", "wb") as fh:
    pickle.dump(gb, fh)

improved = {
    "classifier": "HistGradientBoostingClassifier",
    "features": features7,
    "cv_results": cv_results,
    "improved_punzi_threshold": thr_punzi_imp,
    "signal_efficiency_improved_punzi": sig_eff_imp_punzi,
    "background_efficiency_improved_punzi": bkg_eff_imp_punzi,
}
with open("../improved_results.json", "w") as fh:
    json.dump(improved, fh, indent=2)

with open("../bdt_results.json") as fh:
    bdt_res = json.load(fh)
bdt_res["signal_efficiency_improved_punzi"]     = sig_eff_imp_punzi
bdt_res["background_efficiency_improved_punzi"] = bkg_eff_imp_punzi
bdt_res["improved_punzi_threshold"]             = thr_punzi_imp
with open("../bdt_results.json", "w") as fh:
    json.dump(bdt_res, fh, indent=2)
print("saved improved_model.pkl, improved_results.json, and updated bdt_results.json")